# 05 - Evaluate a stored runn

Grades a run from `experiment_runs` with an LLM judge, it writes to the `evaluation_results`.
The answer model isn't called again, the judge sees the answer and then also the context that tpoduced it.

Three measures, and kept separate so a failure can be located: **recall@k** (did retrieval
find the right chunkss), **correctness / groundedness** for in scope questions, and
**refusal / invented_rule** for near_miss and out_of_scope.

Aa answer can be correct and ungrounded, right by general knowledge but unsupported by
the that is givan context. That combination of this is the interesting one for RQ3.

In [0]:
# Test - new manual checked csv

import csv
user = spark.sql("SELECT current_user()").collect()[0][0]
p = f"/Workspace/Users/{user}/manual_grading_20261031T191325Z-09e6e4.csv"

with open(p, encoding="utf-8") as fh:
    m = list(csv.DictReader(fh))

print(len(m), "rader")
for r in m:
    print(r["question_id"], "|", r["correctness"], "|", r["groundedness"], "|", r["refusal"])

In [0]:
%pip install -q databricks-sdk
dbutils.library.restartPython()

In [0]:
# restartPython() resets sys.path
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
from src.evaluate import (EvalConfig, agreement, blind_template, cohens_kappa,
                          evaluate_run, format_summary, load_run, manual_template,
                          save_evaluations, summarize, unblind)

cfg = EvalConfig()
print("Judge:  ", cfg.judge_endpoint)
print("Runs:   ", cfg.runs_table)
print("Results:", cfg.results_table)

## Pick a run

In [0]:
display(
    spark.table(cfg.runs_table)
         .groupBy("run_id", "run_label", "variant")
         .agg({"question_id": "count", "created_at": "max"})
         .orderBy("max(created_at)", ascending=False)
)

In [0]:
RUN_ID = ""   # paste a run_id, or leave empty for the most recent run

if not RUN_ID:
    RUN_ID = (spark.table(cfg.runs_table)
              .orderBy("created_at", ascending=False)
              .select("run_id").first()[0])

rows = load_run(spark, cfg, RUN_ID)
print(RUN_ID, "->", len(rows), "rows to grade")

## Grade

In [0]:
def progress(position, total, evaluation):
    if evaluation["judge_error"]:
        verdict = f"JUDGE ERROR  {evaluation['judge_error']}"
    elif evaluation["in_scope"]:
        verdict = f"{evaluation['correctness']} / {evaluation['groundedness']}"
    else:
        verdict = f"{evaluation['refusal']}, invented_rule={evaluation['invented_rule']}"
    print(f"[{position}/{total}] {evaluation['question_id']:<10} {verdict}")


evaluations = evaluate_run(spark, cfg, RUN_ID, on_progress=progress)

n_failed = sum(1 for e in evaluations if e["judge_error"])
print(f"\n{len(evaluations)} graded, {n_failed} judge errors")

In [0]:
table = save_evaluations(spark, cfg, evaluations)
print("Saved to", table)

## Short Summary

`correct` and `grounded` are shares of in-scope questions, `declined` and `invented`
of out-of-scope ones. Each row only fills the columns that apply to it.

In [0]:
n_failed = sum(1 for e in evaluations if e["judge_error"])
if n_failed:
    print(f"WARNING: {n_failed} of {len(evaluations)} not graded, excluded below\n")
print(format_summary(summarize(evaluations)))

## What actually went wrong...

Everything that isn't a clean pass, with the judge's reason in mind. The errror analysis in the
report is built from this list, not from the table above.

In [0]:
for e in sorted(evaluations, key=lambda e: e["question_id"]):
    bad = (e["judge_error"]
           or (e["in_scope"] and (e["correctness"] != "correct" or e["groundedness"] != "grounded"))
           or (not e["in_scope"] and (e["refusal"] != "declined" or e["invented_rule"])))
    if not bad:
        continue
    recall = "-" if e["recall_at_k"] is None else f"{e['recall_at_k']:.2f}"
    print(f"{e['question_id']:<10} [{e['question_type']:<20}] recall@k={recall}")
    if e["judge_error"]:
        print(f"           judge error: {e['judge_error']}")
    elif e["in_scope"]:
        print(f"           {e['correctness']} / {e['groundedness']} - {e['judge_reason']}")
    else:
        print(f"           {e['refusal']}, invented_rule={e['invented_rule']} - {e['judge_reason']}")
    print()

## Export for manual grading

Score these by hand **before** looking at the judge's output above, beacuse otherwise the
agreement number can be faulty. Same labels as the judge, allowed values are in the
comments in `src/evaluate.py`.

Lands outside the repo, so half-finished grading isn't committed by accident.

In [0]:
# Cell to be changed later, no overwrites now!!!

import csv
from pathlib import Path

user = spark.sql("SELECT current_user()").collect()[0][0]
manual_path = f"/Workspace/Users/{user}/manual_grading_{RUN_ID}.csv"

if Path(manual_path).exists():
    print("Already graded, not overwriting:", manual_path)
else:
    template = manual_template(rows)
    with open(manual_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=list(template[0]))
        writer.writeheader()
        writer.writerows(template)
    print("Wrote", manual_path)
    print(f"{len(template)} rows to grade by hand")

## Blind manual grading (final run)

A and B are mixed in a random order and the intresting thing is that the variant is hidden. Each grader gets the same file and grades it alone, and this is done before looking at the judges output. The key is closed until all of the gradinf is done. look at: `docs/analysis_plan.md`.

In [0]:
# Blind export for the final run: A and B mixed, variant hidden.
# Each grader gets the same file and grades alone. Don't open the key until all grading is done.
import csv, json
from pathlib import Path

RUN_A = ""   # final run_id for Variant A
RUN_B = ""   # final run_id for Variant B
GRADERS = ["zakaria", "nasser"]   # or just ["zakaria"]

user = spark.sql("SELECT current_user()").collect()[0][0]
folder = Path(f"/Workspace/Users/{user}/manual_final")
folder.mkdir(exist_ok=True)
key_path = folder / "key_DO_NOT_OPEN.json"

if key_path.exists():
    print("Already exported, not overwriting:", folder)
else:
    final_rows = load_run(spark, cfg, RUN_A) + load_run(spark, cfg, RUN_B)
    template, key = blind_template(final_rows, seed=2026)
    for grader in GRADERS:
        with open(folder / f"manual_{grader}.csv", "w", newline="", encoding="utf-8") as fh:
            writer = csv.DictWriter(fh, fieldnames=list(template[0]))
            writer.writeheader()
            writer.writerows(template)
    key_path.write_text(json.dumps(key, indent=2))
    print(len(template), "items per grader in", folder)

### Grader against grader

Only with the two graders. Run when both files are filled in, before discussing. Settle the
listed disagreements together and save the agreed labels as `manual_consensus.csv` in the
same folder. With one grader, and then rename that grader's file to `manual_consensus.csv`.

In [0]:
# Grader against grader, before discussing. Disagreements are listed to settle together.
graded = {g: sorted(csv.DictReader(open(folder / f"manual_{g}.csv", encoding="utf-8")),
                    key=lambda r: r["item_id"]) for g in GRADERS}
first, second = graded[GRADERS[0]], graded[GRADERS[1]]

for field in ("correctness", "groundedness", "refusal"):
    labels_1 = [r[field].strip().lower() for r in first]
    labels_2 = [r[field].strip().lower() for r in second]
    kappa = cohens_kappa(labels_1, labels_2)
    print(f"{field:<14} kappa={'-' if kappa is None else f'{kappa:.2f}'}")
    for row, x, y in zip(first, labels_1, labels_2):
        if x and y and x != y:
            print(f"    {row['item_id']}  {row['question_id']:<10} {x} vs {y}")

### Judge against the agreed manual grades

Per run, against the latest v2 judgment of each question.

In [0]:
# Judge (v2) against the agreed manual grades, per run.
consensus = list(csv.DictReader(open(folder / "manual_consensus.csv", encoding="utf-8")))
key = json.loads(key_path.read_text())

for run_id, manual in unblind(consensus, key).items():
    judged = [r.asDict() for r in spark.table(cfg.results_table)
              .filter(f"run_id = '{run_id}' AND judge_version = 'v2'").collect()]
    latest = {e["question_id"]: e for e in sorted(judged, key=lambda e: e["created_at"])}

    print("\n", run_id)
    for field, result in agreement(list(latest.values()), manual).items():
        if not result["n"]:
            continue
        kappa = "-" if result["kappa"] is None else f"{result['kappa']:.2f}"
        print(f"  {field:<14} n={result['n']:<3} exact={result['exact_match']:.2f}  kappa={kappa}")

## Judge against manual grading

Run once the CSV is filled in. `kappa` is agreement corrected for chance, above the ~0.6
usually reads as substantial. Below that then the judges numbers need a that the judge's numbers need a condition, and the
disagreements are worth going through one by one. often the prompt is not clear rather than the judge wrong.

In [0]:
with open(manual_path, encoding="utf-8") as fh:
    manual = list(csv.DictReader(fh))

for field, result in agreement(evaluations, manual).items():
    if not result["n"]:
        continue
    kappa = "-" if result["kappa"] is None else f"{result['kappa']:.2f}"
    print(f"{field:<14} n={result['n']:<3} exact={result['exact_match']:.2f}  kappa={kappa}")
    for d in result["disagreements"]:
        print(f"    {d['question_id']:<10} judge={d['judge']:<12} manual={d['manual']}")